In [31]:
# Q1 Embedding a query
import numpy as np
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

v = embed.encode(q)
print(" The first value v[0] is:", v[0])

 The first value v[0] is: -0.020582034180917443


In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
doc = next(doc for doc in documents if '07-sqlitesearch-vector' in doc['filename'])

# embed documents
d = embed.encode(doc['content'])

# Q2: Cosine similarity 
score = np.dot(d,v)
print(score)

In [ ]:
# Q# chunking and searching by hand 
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4: We get {len(chunks)} chunks")
# embbed chunk's content with encode barch 

In [20]:
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

In [ ]:
score = X.dot(v)
print(score)

In [ ]:
idx = np.argmax(score)
idx, score[idx]
idx, score[idx], chunks[idx]

In [ ]:
# top 5 
top5 = np.argsort(-score)[:5]
print(top5)

In [ ]:
for idx in top5: 
    print(score[idx])
    print(chunks[idx])
    print()

In [14]:
# vector search with minsearch 
from minsearch import VectorSearch

# 1. create an index 
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

# search 
query = " What metric do we use to evaluate a search engine??" 

query_vector = embed.encode(query)
results = vindex.search(query_vector, num_results=5)
results

NameError: name 'X' is not defined

In [17]:
# Q5: keyword search 
from minsearch import Index

# Step 1: define the structure (what fields, what type)
index = Index(
    text_fields=["content"])

query_compare = "How do I store vectors in PostgreSQL?"

# Step 2: load the actual data
index.fit(chunks)

# search with filters and boosts 
text_results = index.search(query_compare,  num_results=5) 
text_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [ ]:
query_vector = embed.encode(query_compare)
vector_results = vindex.search(query_vector, num_results=5)
vector_results

In [ ]:
# Q6: Hybrid Search:  Reciprocal Rank Fusion (RRF)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query_rrf = "How do I give the model access to tools?"

# search with filters and boosts 
text_results = index.search(query_rrf,  num_results=5) 

query_vector = embed.encode(query_rrf)
vector_results = vindex.search(query_vector, num_results=5)

results = rrf([vector_results, text_results])

In [ ]:
results

In [138]:
# HOMEWORK - EVALUATION
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [139]:
# get chunks 
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [140]:
# evaluating search 
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [141]:
import os 
import json
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

class Questions(BaseModel): # forced output 
    questions: list[str]

In [142]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "content": q,
            "document": doc["filename"]
        })

    return results, usage

In [143]:
# first 5 documents only 
from tqdm.auto import tqdm
from evaluation_utils import calc_total_price

ground_truth_3 = []
usages_3 = []

for doc in tqdm(documents[:3]): # first first documents 
    records, usage = generate_ground_truth(doc)
    ground_truth_3.extend(records)
    usages_3.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [146]:
ground_truth_3

[{'content': 'What are the main limitations of Large Language Models?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'content': 'How does a Retrieval-Augmented Generation system help with the limitations of LLMs?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'content': 'What kind of project will we be working on in this module?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'content': 'What is the goal of Part 1 of this module?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'content': 'How does a Large Language Model make predictions?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'content': 'What are the prerequisites for this module?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'content': 'How do I install uv on Mac or Linux?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'content': 'What dependencies do we need to add to the project?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'co

In [ ]:
# Q! average number of input tokens across the 3 calls 
sum(u.input_tokens for u in usages_3) / len(usages_3)

1381.6666666666667

In [145]:
calc_total_price(usages_3)

0.0005368700000000001

In [ ]:
ground_truth = []
usages = []

for doc in tqdm(documents):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

In [ ]:
# the ground truth 
import pandas as pd 

df_ground_truth = pd.read_csv("data/ground-truth.csv")
df_ground_truth.head(-1)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md
...,...,...
354,What features are included in the finished fit...,07-project-example/lessons/06-summary.md
355,How should I break up a long article or transc...,07-project-example/lessons/07-chunking.md
356,"When I have several blog posts or wiki pages, ...",07-project-example/lessons/07-chunking.md
357,"For a single long PDF or video transcript, wha...",07-project-example/lessons/07-chunking.md


In [134]:
from minsearch import Index, VectorSearch
# text index 
def text_search(query): 
    boost_dict = {"filename": 3.0, "content": 0.5}
    
    index = Index(text_fields=["content"])
    index.fit(chunks)
    results = index.search(query, boost_dict=boost_dict, num_results=10) 
    return results

In [135]:
# vector index
def vector_search(query): 

    vindex = VectorSearch(keyword_fields=["content"])
    
    vindex.fit(X, chunks)
    query_vector = embed.encode(query)
    results = vindex.search(query_vector, num_results=10)
    
    return results

In [128]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query)
    vector_results = vector_search(query)
    return rrf([text_results, vector_results], k=k)

In [82]:
ground_truth = df_ground_truth.to_dict(orient="records")
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [ ]:
# Q2: first result with text search 
result =  text_search(q)
result[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [ ]:
# Q3: First result with vector search 
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

result =  vector_search(q)
result[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [77]:
# general function for search engine
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
# Q4: Evaluation metric text search 
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [ ]:
# Q5: Evaluation metric vector search 
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.5646472663139328}

In [ ]:
# Q6: evaluation - Tuning hybrid search 
from functools import partial

for k in [1, 50, 100, 200]:
    search_fn = partial(hybrid_search, k=k)
    result = evaluate(ground_truth, search_fn)
    print(f"k={k} -> MRR: {result['mrr']:.4f}, Hit Rate: {result['hit_rate']:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1 -> MRR: 0.6482, Hit Rate: 0.8389


  0%|          | 0/360 [00:00<?, ?it/s]

k=50 -> MRR: 0.6379, Hit Rate: 0.8361


  0%|          | 0/360 [00:00<?, ?it/s]

k=100 -> MRR: 0.6379, Hit Rate: 0.8361


  0%|          | 0/360 [00:00<?, ?it/s]

k=200 -> MRR: 0.6379, Hit Rate: 0.8361
